In [1]:
# ── CELL 1: LOAD THE DATA ─────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# read the CSV into a DataFrame (a table in memory)
df = pd.read_csv("data/telco_churn.csv")

# shape = (rows, columns). we expect roughly (7043, 21)
print("Shape:", df.shape)

# see the first 5 rows to eyeball the real data
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Good instinct — always know *what* and *why* before writing code. Here's the full picture.

**The problem in plain words**

A telecom company (think Jio, Airtel) makes money from monthly subscriptions. Every month, some customers cancel and leave — this is called **churn**. Losing a customer is expensive: the company spent money acquiring them, and a lost customer means lost future revenue. It's far cheaper to *keep* an existing customer than to find a new one.

So the company wants to know, in advance: **which customers are likely to leave soon?** If they can spot them early, they can act — offer a discount, a better plan, a loyalty perk — and stop them from leaving.

**What we're going to do**

We build a model that learns from ~7,000 past customers (where we already know who left and who stayed) to spot the patterns of a "likely leaver." Then it can score a *current* customer and say "this person has a 78% chance of churning."

Concretely, for each customer we look at things like:
- **tenure** — how long they've been a customer (new customers leave more)
- **Contract** — month-to-month vs 1-year vs 2-year (month-to-month churns far more)
- **MonthlyCharges** — how much they pay
- **services** — internet type, tech support, streaming, etc.

The model finds which combinations lead to leaving. This is **binary classification** — sorting each customer into "will churn" (1) or "will stay" (0). Exactly the same problem type as loan-default, just a friendlier domain.

**The real-world payoff (your interview line)**

> "The business value is retention: if the model flags a customer as high-risk, the company proactively offers them a deal. Keeping a customer is cheaper than acquiring a new one, so even catching *some* churners saves real money."

That's why churn models are everywhere — telecom, streaming (Netflix), SaaS, gyms, banks. It's one of the most common ML jobs in industry, which makes it a strong portfolio piece.

**How it differs from loan-default (what makes it worth doing)**

```
SAME:  binary classification, imbalance, precision/recall/AUC
NEW:   • built with scikit-learn (production tools, not from scratch)
       • lots of TEXT columns to encode into numbers
       • SHAP to explain WHY a customer is flagged
       • friendlier, easier-to-explain business story
```

So the point of this project isn't to re-learn the ML concepts — you already own those. It's to show you can **build the production way with real tools** and **explain the business clearly** — the two things that make churn a perfect second project.

Make sense? If so, run Cell 2 and tell me the churn split and whether `TotalCharges` shows as text or numeric.

In [2]:
# ── CELL 2: UNDERSTAND THE TARGET AND COLUMN TYPES ────────────────────────────

# our target is "Churn" — did the customer leave? (Yes / No)
# check the balance: how many left vs stayed
print("=== Churn counts ===")
print(df["Churn"].value_counts())
print()
print("=== Churn as fractions ===")
print(df["Churn"].value_counts(normalize=True))
print()

# df.info() shows every column, its non-null count, and its type
# object = text column (needs encoding), int64/float64 = already numeric
print("=== Column types (object = text, needs encoding) ===")
df.info()

=== Churn counts ===
Churn
No     5174
Yes    1869
Name: count, dtype: int64

=== Churn as fractions ===
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

=== Column types (object = text, needs encoding) ===
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport  

Two important findings here — let's read them.
1. The imbalance is mild — 73% stayed, 27% churned. Much gentler than loan-default's 93/7. Accuracy is still not the whole story (a "predict everyone stays" model would score 73%), but this dataset is friendlier — models will actually catch churners without heavy threshold tuning.
2. TotalCharges is str (text), not numeric — the sneaky one I warned about. Look:
19  TotalCharges   7043 non-null   str    ← should be a number!
It looks like money (29.85, 1889.5) but pandas read it as text. The reason: a handful of rows have a blank space " " instead of a number (these are brand-new customers with tenure 0 who haven't been charged yet). That one space forces the whole column to text. We must fix this before modeling.
Also note: 18 text columns to encode. That's the bulk of the new work — but it's mechanical, not hard.

In [3]:
# ── CELL 3: CLEAN THE DATA ────────────────────────────────────────────────────

# ── Fix 1: TotalCharges is text but should be numeric ─────────────────────────
# pd.to_numeric converts text to numbers.
# errors="coerce" means: if a value can't convert (like the blank " "),
# turn it into NaN (missing) instead of crashing.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# how many blanks did that create?
print("Missing TotalCharges after conversion:", df["TotalCharges"].isnull().sum())

# these are new customers (tenure 0) never charged yet — fill with 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# ── Fix 2: drop customerID — an ID is not a useful feature ─────────────────────
# it's just a random label, tells the model nothing about churn
df = df.drop("customerID", axis=1)

# ── verify ────────────────────────────────────────────────────────────────────
print("TotalCharges type now:", df["TotalCharges"].dtype)   # should be float64
print("Shape after dropping ID:", df.shape)                 # (7043, 20)
print("Total missing values:", df.isnull().sum().sum())     # should be 0

Missing TotalCharges after conversion: 11
TotalCharges type now: float64
Shape after dropping ID: (7043, 20)
Total missing values: 0


Clean. Exactly as predicted:

11 missing values appeared — those are the new customers (tenure 0) with a blank space instead of a charge. Now filled with 0, which makes sense (they haven't been charged yet).
TotalCharges is now float64 — the sneaky text-that-should-be-a-number is fixed.
20 columns (dropped the useless ID), 0 missing — ready to go.

That TotalCharges fix is a great interview detail, by the way: "I caught a column that looked numeric but was stored as text because of hidden blank spaces — a common real-world data trap." Shows you inspect data carefully.
Now the core new skill of this project — encoding text into numbers.
Why we need it: models do math, and you can't multiply "Month-to-month" by a weight. So we convert every text column into numbers. There are two kinds:

Binary text (2 options)      → map to 0/1
  e.g. gender: Female/Male → 0/1
       Churn: No/Yes → 0/1

Multi-category text (3+)     → "one-hot encoding" (one column each)
  e.g. Contract: Month-to-month / One year / Two year
       → becomes 3 yes/no columns

In [4]:
# ── CELL 4: START ENCODING — THE TARGET FIRST ─────────────────────────────────

# our target Churn is text ("Yes"/"No"). map it to 1/0.
# 1 = churned (left), 0 = stayed
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Churn is now numeric:")
print(df["Churn"].value_counts())
print()

# ── let's see which columns are still text and need encoding ───────────────────
# select_dtypes(include="object") picks only the text columns
# (in your pandas version they may show as 'str' — same idea)
text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()

print(f"Text columns still needing encoding ({len(text_columns)}):")
for col in text_columns:
    # for each, show how many unique values it has
    n_unique = df[col].nunique()
    print(f"  {col}: {n_unique} unique values -> {list(df[col].unique())}")

Churn is now numeric:
Churn
0    5174
1    1869
Name: count, dtype: int64

Text columns still needing encoding (15):
  gender: 2 unique values -> ['Female', 'Male']
  Partner: 2 unique values -> ['Yes', 'No']
  Dependents: 2 unique values -> ['No', 'Yes']
  PhoneService: 2 unique values -> ['No', 'Yes']
  MultipleLines: 3 unique values -> ['No phone service', 'No', 'Yes']
  InternetService: 3 unique values -> ['DSL', 'Fiber optic', 'No']
  OnlineSecurity: 3 unique values -> ['No', 'Yes', 'No internet service']
  OnlineBackup: 3 unique values -> ['Yes', 'No', 'No internet service']
  DeviceProtection: 3 unique values -> ['No', 'Yes', 'No internet service']
  TechSupport: 3 unique values -> ['No', 'Yes', 'No internet service']
  StreamingTV: 3 unique values -> ['No', 'Yes', 'No internet service']
  StreamingMovies: 3 unique values -> ['No', 'Yes', 'No internet service']
  Contract: 3 unique values -> ['Month-to-month', 'One year', 'Two year']
  PaperlessBilling: 2 unique values -> ['Yes'

Perfect — now we can see exactly what we're dealing with. 15 text columns, and they split into two clear groups:
BINARY (2 values) → simple 0/1 mapping:
  gender, Partner, Dependents, PhoneService, PaperlessBilling

MULTI-CATEGORY (3-4 values) → one-hot encoding:
  MultipleLines, InternetService, OnlineSecurity, OnlineBackup,
  DeviceProtection, TechSupport, StreamingTV, StreamingMovies,
  Contract, PaymentMethod
One thing to notice: several columns have "No internet service" or "No phone service" as a third value — these are really just "No" in disguise (if you have no internet, you obviously have no online security). We could simplify them, but one-hot encoding handles them fine as-is, so we'll keep it simple.
The clean way to do all this: pd.get_dummies() — one function that one-hot encodes everything automatically. It's the standard, production approach (and much less error-prone than mapping 15 columns by hand).

In [7]:
# ── CELL 5 (FIXED): ENCODE TEXT COLUMNS WITHOUT DAMAGING DECIMALS ──────────────

# one-hot encode all text columns (same as before)
df_encoded = pd.get_dummies(df, drop_first=True)

# THE FIX:
# get_dummies creates True/False columns for the encoded features.
# we only want to convert THOSE to 1/0 — NOT the decimal money columns.
# select_dtypes(include="bool") picks only the True/False columns.
bool_columns = df_encoded.select_dtypes(include="bool").columns
df_encoded[bool_columns] = df_encoded[bool_columns].astype(int)

# now verify: decimals should be preserved this time
print("Charge columns (decimals should be intact now):")
print(df_encoded[["MonthlyCharges", "TotalCharges"]].head())
print()
print("Shape:", df_encoded.shape)
print()
# confirm all columns are numeric (int or float, no text)
print("Column types:")
print(df_encoded.dtypes.value_counts())

Charge columns (decimals should be intact now):
   MonthlyCharges  TotalCharges
0           29.85         29.85
1           56.95       1889.50
2           53.85        108.15
3           42.30       1840.75
4           70.70        151.65

Shape: (7043, 31)

Column types:
int64      29
float64     2
Name: count, dtype: int64


Let me slow right down and explain this clearly, because understanding your data is more important than rushing ahead.

**What a DataFrame actually is**

Think of it as an Excel sheet:
- **Rows** = one per customer. You have 7,043 rows = 7,043 customers.
- **Columns** = one per piece of information about that customer.

So each row is *one person*, and reading across that row tells you everything about them: their gender, how long they've been a customer, what they pay, and whether they left.

**What your columns mean (in plain words)**

Here are the original columns and what each tells you about a customer:

```
gender            → Male or Female
SeniorCitizen     → is the person a senior? (0 = no, 1 = yes)
Partner           → do they have a partner? (Yes/No)
Dependents        → do they have kids/dependents? (Yes/No)
tenure            → how many MONTHS they've been a customer
PhoneService      → do they have phone service? (Yes/No)
MultipleLines     → multiple phone lines? (Yes/No/No phone)
InternetService   → type of internet (DSL / Fiber optic / None)
OnlineSecurity    → do they pay for online security? (Yes/No)
OnlineBackup      → online backup add-on? (Yes/No)
DeviceProtection  → device protection add-on? (Yes/No)
TechSupport       → tech support add-on? (Yes/No)
StreamingTV       → streaming TV? (Yes/No)
StreamingMovies   → streaming movies? (Yes/No)
Contract          → contract length (Month-to-month / One year / Two year)
PaperlessBilling  → paperless billing? (Yes/No)
PaymentMethod     → how they pay (electronic check, mailed check, etc.)
MonthlyCharges    → how much they pay per month (dollars)
TotalCharges      → total they've paid over their whole time (dollars)
Churn             → did they LEAVE? (this is what we predict) Yes/No
```

Reading one customer (row 0): *Female, not senior, has a partner, no kids, been a customer 1 month, has phone service, DSL internet, month-to-month contract, pays $29.85/month, and... did not churn.*

**What we just did to the columns (the encoding)**

Models can't read words like "Fiber optic" — they only do math on numbers. So we converted every text column into number columns. That's why your columns grew from **20 to 31**.

Here's exactly what happened, using `Contract` as the example:

```
BEFORE — one text column:
   Contract
   "Month-to-month"
   "One year"
   "Two year"

AFTER — two yes/no number columns:
   Contract_One year    Contract_Two year
        0                    0              ← means Month-to-month
        1                    0              ← means One year
        0                    1              ← means Two year
```

So "Contract" became two columns of 0s and 1s. A customer with **0 in both** is Month-to-month (the "baseline" — we dropped it to avoid repeating information). Same thing happened to `PaymentMethod` (became several columns), `InternetService`, and all the other multi-option columns. That splitting is why you gained 11 extra columns.

The simple ones (like `gender`, `Partner`) just became a single 0/1 column each.

**The one thing I want you to check** (the subtle bug I mentioned)

We ran `.astype(int)` which turned everything into whole numbers. That's fine for the 0/1 columns — but `MonthlyCharges` and `TotalCharges` are *dollars with decimals* ($29.85). Turning them into `int` chops off the decimals → $29.85 becomes $29. Let's look:

```python
# check: did .astype(int) damage the decimal columns?
print(df_encoded[["MonthlyCharges", "TotalCharges"]].head())
```

Run that one line and tell me — do you see `29` or `29.85` for MonthlyCharges? If it shows `29` (no decimals), we chopped them, and I'll show you the clean one-line fix. This is a real lesson in being careful with blanket conversions.

In [8]:
# ── CELL 6: SPLIT AND SCALE (using scikit-learn) ──────────────────────────────

# scikit-learn gives us ready-made tools instead of coding these by hand
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── separate features (X) from target (y) ─────────────────────────────────────
X = df_encoded.drop("Churn", axis=1)   # all columns EXCEPT churn = the inputs
y = df_encoded["Churn"]                # just the churn column = the answer

print("Features shape:", X.shape)      # (7043, 30)
print("Target shape:  ", y.shape)      # (7043,)

# ── train/test split ──────────────────────────────────────────────────────────
# train_test_split does the shuffle + split for us in one line.
#   test_size=0.2      → 20% for testing, 80% for training
#   random_state=42    → fixed seed, reproducible (same as our np.random.seed)
#   stratify=y         → KEEP the 73/27 churn ratio in BOTH sets (important!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTraining customers:", X_train.shape[0])   # ~5634
print("Test customers:    ", X_test.shape[0])      # ~1409

# ── scaling ───────────────────────────────────────────────────────────────────
# StandardScaler rescales features so they're comparable.
# unlike our manual min-max (0 to 1), this uses "standardization":
# it centers each feature to mean 0 and scales to standard deviation 1.
# it's the standard choice and handles outliers a bit better than min-max.
scaler = StandardScaler()

# fit on TRAINING data only (learn the mean/std), then transform both
# — same anti-leakage rule you already know
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform in one call
X_test_scaled  = scaler.transform(X_test)        # transform only (no fit!)

print("\nScaling done. Data ready for modeling.")
print("X_train_scaled shape:", X_train_scaled.shape)

Features shape: (7043, 30)
Target shape:   (7043,)

Training customers: 5634
Test customers:     1409

Scaling done. Data ready for modeling.
X_train_scaled shape: (5634, 30)


A few things worth noticing, since they're the "production tool" versions of what you built by hand:

train_test_split does in one line what took you several cells in loan-default (shuffle + seed + slice).
stratify=y is a nice touch — it forces both train and test to keep the exact 73/27 churn ratio, so neither set is accidentally lopsided. You checked this manually before; sklearn guarantees it.
StandardScaler is a different scaling method (mean 0, std 1) instead of min-max. It's the common default and handles outliers better — which, remember, was min-max's weakness in loan-default. Same anti-leakage rule: fit on train only.

Run it and tell me the shapes — you're looking for ~5,634 training and ~1,409 test customers. Then we build our first model, and you'll see how little code scikit-learn needs.

In [9]:
# ── CELL 7: TRAIN LOGISTIC REGRESSION (scikit-learn) ──────────────────────────

from sklearn.linear_model import LogisticRegression

# in loan-default you wrote sigmoid + cost + gradients + training loop by hand.
# scikit-learn does ALL of that internally. watch:

# create the model
#   max_iter=1000     → how many gradient descent steps (like your iterations)
#   class_weight="balanced" → tells it to pay more attention to the rare
#                              churn class, countering the 73/27 imbalance
model_lr = LogisticRegression(max_iter=1000, class_weight="balanced")

# train it — this ONE line replaces your entire Week 3 training loop
model_lr.fit(X_train_scaled, y_train)

# predict on the test set
y_pred = model_lr.predict(X_test_scaled)               # 0/1 predictions
y_prob = model_lr.predict_proba(X_test_scaled)[:, 1]   # probability of churn

print("Model trained! First 10 predicted probabilities:")
print(y_prob[:10].round(3))

Model trained! First 10 predicted probabilities:
[0.118 0.852 0.141 0.665 0.061 0.805 0.7   0.294 0.009 0.652]


wo lines of real work: LogisticRegression(...) and .fit(...). That's the whole model. Everything you built by hand in Week 3 is happening inside .fit().
Two things worth understanding:

class_weight="balanced" — this is sklearn's built-in fix for imbalance. Instead of tuning the threshold afterward like you did in loan-default, it tells the model during training to weight the rare churn class more heavily, so it doesn't just predict "everyone stays." A cleaner, more professional way to handle imbalance.
.predict_proba()[:, 1] — gives the probability of churn (the [:, 1] picks the "churn" class). Same probabilities you computed with your sigmoid, just handed to you.

Run it and tell me what the first 10 probabilities look like. Then we'll evaluate it properly — and here's the good part: we get to reuse your understanding of precision, recall, and AUC, except sklearn computes them in one line each.


Look at those probabilities — this is already a far healthier model than loan-default's logistic regression. Remember how that one couldn't push past 0.13 for anyone? This one is confidently spreading across the full range: 0.009 (very safe), 0.852 (very likely to churn), 0.665, 0.294. It's actually distinguishing customers.
Why the difference? Two reasons: this data has no extreme-outlier scaling problem (StandardScaler handled it cleanly), and class_weight="balanced" freed the model to make confident churn predictions. The tools did their job.
Now we evaluate — and here's where your loan-default knowledge pays off. You already know what precision, recall, and AUC mean; sklearn just computes them instantly.

In [10]:
# ── CELL 8: EVALUATE (scikit-learn computes your Week 4 metrics instantly) ─────

from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score
)

# ── the confusion matrix — you built this by hand; sklearn does it in one call ─
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print("                 Predicted Stay   Predicted Churn")
print(f"Actual Stay        {cm[0][0]:>6}          {cm[0][1]:>6}")
print(f"Actual Churn       {cm[1][0]:>6}          {cm[1][1]:>6}")
print()

# ── classification_report — precision, recall, F1 for BOTH classes at once ─────
# this single function replaces all the metric functions you hand-coded!
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Stay", "Churn"]))

# ── AUC — the ranking metric you built from scratch with trapezoids ────────────
auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC: {auc:.4f}")

Confusion Matrix:
                 Predicted Stay   Predicted Churn
Actual Stay           749             286
Actual Churn           80             294

Classification Report:
              precision    recall  f1-score   support

        Stay       0.90      0.72      0.80      1035
       Churn       0.51      0.79      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

ROC-AUC: 0.8414


This is a genuinely good result — and much more useful out-of-the-box than your loan-default model. Let me read it with you.
The headline: AUC = 0.8414. That's a solid churn model (above 0.80 is the respectable zone). But the confusion matrix tells the real story:
                 Predicted Stay   Predicted Churn
Actual Stay          749              286
Actual Churn          80              294   ← caught 294 of 374 churners!
Read the Churn row — this is what matters:

Caught 294 of 374 real churners = 79% recall. Out of everyone who actually left, the model flagged nearly 4 out of 5. Compare that to loan-default, which caught zero at its default threshold. This model works straight away.
Missed only 80 churners (bottom-left) — the ones who slipped through.
286 false alarms (top-right) — customers flagged as churn risk who'd have stayed anyway.

Precision for Churn = 0.51 — when the model flags someone, it's right about half the time. That sounds mediocre, but for churn it's fine, and here's the business reasoning (your interview answer):

"51% precision means half our retention offers go to people who'd have stayed anyway. But a retention offer (a discount) is cheap, while losing a customer is expensive. So we happily accept some false alarms to catch 79% of real churners. I'd tune the exact trade-off to the cost of the offer versus the value of a customer."

That's the same precision/recall trade-off from loan-default, now with a clean business story.
The one thing to notice — why recall is already high: that class_weight="balanced" did its job. Instead of you manually lowering a threshold like in loan-default, the model was trained to take the rare churn class seriously from the start. Cleaner, more professional.
Where you are:
✅ Data cleaned & encoded
✅ Split & scaled with sklearn
✅ Logistic Regression: AUC 0.841, 79% churn recall
Now let's do what makes this a comparison project — train two more models and see who wins. This is fast in sklearn.

In [11]:
# ── CELL 9: TRAIN RANDOM FOREST AND XGBOOST ───────────────────────────────────

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ── Random Forest ─────────────────────────────────────────────────────────────
# you built one from scratch in loan-default! sklearn's version in 2 lines.
# n_estimators=200 = 200 trees; class_weight handles the imbalance
model_rf = RandomForestClassifier(
    n_estimators=200, class_weight="balanced", random_state=42
)
model_rf.fit(X_train_scaled, y_train)
rf_prob = model_rf.predict_proba(X_test_scaled)[:, 1]

# ── XGBoost ───────────────────────────────────────────────────────────────────
# the "gradient boosting" model that wins most tabular ML competitions.
# it builds trees SEQUENTIALLY, each fixing the previous one's mistakes.
# scale_pos_weight handles imbalance (ratio of stay:churn ≈ 73:27 ≈ 2.7)
model_xgb = XGBClassifier(
    n_estimators=200, scale_pos_weight=2.7,
    random_state=42, eval_metric="logloss"
)
model_xgb.fit(X_train_scaled, y_train)
xgb_prob = model_xgb.predict_proba(X_test_scaled)[:, 1]

# ── compare all three by AUC ──────────────────────────────────────────────────
print("═══ AUC COMPARISON — ALL THREE MODELS ═══")
print(f"  Logistic Regression: {roc_auc_score(y_test, y_prob):.4f}")
print(f"  Random Forest:       {roc_auc_score(y_test, rf_prob):.4f}")
print(f"  XGBoost:             {roc_auc_score(y_test, xgb_prob):.4f}")

═══ AUC COMPARISON — ALL THREE MODELS ═══
  Logistic Regression: 0.8414
  Random Forest:       0.8270
  XGBoost:             0.8092


Interesting result — and a genuinely useful one, because it's the opposite of what most people expect. Let's read it honestly:
Logistic Regression: 0.8414   ← the WINNER
Random Forest:       0.8270
XGBoost:             0.8092
The simplest model won. That surprises people — everyone assumes XGBoost always wins because it dominates Kaggle competitions. But here, plain logistic regression beat both tree models. This is actually a great thing to understand and explain.
Why did the simple model win here?

Churn relationships are fairly linear. Things like "longer tenure → less churn" and "higher charges → more churn" are mostly straight-line relationships. Logistic regression is built for linear patterns, so it fits this data naturally.
Small dataset (7,000 rows). XGBoost and Random Forest are powerful but data-hungry — they shine on large, complex datasets with intricate non-linear patterns. On a small, mostly-linear dataset, that power isn't needed and can even slightly overfit.
XGBoost likely needs tuning. Out-of-the-box, its default settings aren't optimal here. With hyperparameter tuning it would likely improve — but the point stands.

The interview lesson (this is gold):

"I tried logistic regression, random forest, and XGBoost. Surprisingly, logistic regression won at 0.841. This taught me that the fanciest model isn't always best — for a small dataset with mostly linear relationships, a simple model can beat XGBoost. I always start simple and only add complexity if it earns its keep."

That answer shows maturity. Junior candidates reach for XGBoost reflexively; you can explain when simple beats complex. Interviewers love that.
So our champion is Logistic Regression (AUC 0.841, 79% churn recall). Clean, interpretable, and the best performer.
Now the coolest part — SHAP, to answer the question every business actually asks: "WHY do customers churn? What should we fix?" This turns your model from a black box into business insight.

In [12]:
# ── CELL 10: WHAT DRIVES CHURN? (feature importance) ──────────────────────────

# for a linear model like logistic regression, the WEIGHTS directly tell us
# each feature's influence:
#   large positive weight → pushes churn UP
#   large negative weight → pushes churn DOWN (protective)

# get the feature names and their learned weights (coefficients)
feature_names = X.columns
coefficients = model_lr.coef_[0]

# pair them and sort by strength
importance = pd.DataFrame({
    "feature": feature_names,
    "weight": coefficients
}).sort_values("weight", ascending=False)

print("═══ TOP 5 FEATURES THAT INCREASE CHURN ═══")
print(importance.head(5).to_string(index=False))
print()
print("═══ TOP 5 FEATURES THAT REDUCE CHURN (protective) ═══")
print(importance.tail(5).to_string(index=False))

═══ TOP 5 FEATURES THAT INCREASE CHURN ═══
                    feature   weight
InternetService_Fiber optic 0.808897
               TotalCharges 0.494748
        StreamingMovies_Yes 0.285948
            StreamingTV_Yes 0.273209
          MultipleLines_Yes 0.201165

═══ TOP 5 FEATURES THAT REDUCE CHURN (protective) ═══
           feature    weight
OnlineSecurity_Yes -0.114503
 Contract_One year -0.296706
 Contract_Two year -0.617746
    MonthlyCharges -1.039440
            tenure -1.159048


This is the payoff — your model just handed you a business strategy. Let me translate these numbers into plain English, because this is exactly what you'd present to a company (and an interviewer).

**What drives customers AWAY (positive weights = more churn):**
```
Fiber optic internet   +0.81   biggest churn driver
TotalCharges           +0.49
Streaming Movies/TV    +0.28   
Multiple phone lines   +0.20
```

**What KEEPS customers (negative weights = protective):**
```
tenure                 -1.16   strongest loyalty factor
MonthlyCharges         -1.04
Two-year contract      -0.62
One-year contract      -0.30
Online security        -0.11
```

**The story these numbers tell:**

- **Tenure is the #1 protector (-1.16).** The longer someone's been a customer, the less likely they leave. New customers are the flight risk. → *Business action: focus retention on customers in their first few months.*

- **Long contracts massively reduce churn** (two-year -0.62, one-year -0.30). Month-to-month customers churn far more. → *Business action: incentivize customers onto longer contracts with discounts.*

- **Fiber optic is the biggest churn driver (+0.81).** This is a fascinating, non-obvious finding — fiber customers leave *more*. Likely reasons: fiber is premium/expensive, so expectations are high and competition is fierce. → *Business action: investigate fiber pricing and service quality.*

**One subtlety worth flagging (shows you think critically):** notice `MonthlyCharges` is protective (-1.04) but `TotalCharges` drives churn (+0.49). That seems contradictory. The likely explanation: `TotalCharges` is tangled up with tenure (long-time customers have high total charges), so the model splits the signal in complex ways. This is **multicollinearity** — the same thing you saw in loan-default's correlated past-due columns. Good thing to mention rather than over-claim.

**Your interview-ready insight:**
> "The model revealed the biggest churn risks are new customers on month-to-month contracts, and surprisingly, fiber optic users. The biggest protectors are long tenure and multi-year contracts. So my recommendation to the business: push longer contracts and investigate fiber pricing."

That sentence turns a model into a *business recommendation* — which is what separates a data scientist from someone who just runs code.

**Where you are:**
```
✅ Three models compared (LogReg wins, AUC 0.841)
✅ 79% of churners caught
✅ Feature importance → real business insights
```

You're basically done with the modeling. What's left is packaging: **a Streamlit app + README + deploy**, same as loan-default. 

Want to build the deployment now, or pause here? Also — quick question that affects the app: should we keep it simple (predict churn from a customer's details) like the loan app, or would you like me to show you one modern addition — SHAP force plots that explain *each individual prediction*? Your call.